In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "00-foundations/transformers/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# The Original Transformer — Exercises

You implement the **five core pieces**; everything else (feed-forward, encoder layer,
full model, training loop) is provided. Each task has a `# YOUR CODE HERE` gap and a
**test cell** below it that tells you when it's correct. Then the whole thing trains.

Order: `attention` → `MultiHead` → `PositionalEncoding` → `causal_mask` → `DecoderLayer`.

If stuck, the solution is `01_transformer_walkthrough.ipynb`. Run this top to bottom;
later cells depend on earlier ones.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## Task 1 — Scaled dot-product attention

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

Return **`(output, weights)`**.
- `Q, K, V`: `(B, H, T, d_k)`.
- Score = `Q @ Kᵀ` scaled by `1/√d_k` → `(B, H, Tq, Tk)`.
- Where `mask == 0`, set the score to `float('-inf')` **before** softmax.
- Softmax over the **last** dim; multiply by `V`.

Hint: transpose the last two dims of `K` with `K.transpose(-2, -1)`; use `masked_fill`.

In [ ]:
def attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    # YOUR CODE HERE
    # 1. scores = ...
    # 2. if mask is not None: scores = scores.masked_fill(...)
    # 3. weights = F.softmax(...)
    # 4. return weights @ V, weights
    raise NotImplementedError

In [ ]:
# TEST — Task 1
B, H, T, d_k = 2, 3, 4, 8
Q, K, V = torch.randn(B, H, T, d_k), torch.randn(B, H, T, d_k), torch.randn(B, H, T, d_k)
out, w = attention(Q, K, V)
assert out.shape == (B, H, T, d_k), f"output shape {out.shape}"
assert w.shape == (B, H, T, T), f"weights shape {w.shape}"
assert torch.allclose(w.sum(-1), torch.ones(B, H, T)), "each row of weights must sum to 1"

cm = torch.tril(torch.ones(1, T, T)).long().unsqueeze(1)
_, wm = attention(Q, K, V, cm)
upper = torch.triu(torch.ones(T, T), diagonal=1).bool()
assert (wm[..., upper] == 0).all(), "masked (future) positions must get 0 weight"

Kc = torch.zeros(B, H, T, d_k); Vc = torch.randn(B, H, T, d_k)
oc, _ = attention(torch.randn(B, H, T, d_k), Kc, Vc)     # identical keys -> uniform avg
assert torch.allclose(oc, Vc.mean(2, keepdim=True).expand(-1, -1, T, -1), atol=1e-5)
print("PASS — attention")

## Task 2 — Multi-head attention (`forward`)

The `__init__` (four `nn.Linear` projections) is given. Implement `forward`:
1. Project `q, k, v` with `W_q, W_k, W_v`.
2. Reshape each `(B, T, d_model)` → `(B, h, T, d_k)`. Hint:
   `W(x).view(B, -1, self.h, self.d_k).transpose(1, 2)`.
3. If `mask is not None`, add a head axis: `mask = mask.unsqueeze(1)`.
4. Call `attention(...)`.
5. Concatenate heads back: `.transpose(1, 2).contiguous().view(B, -1, self.h * self.d_k)`.
6. Apply `self.W_o`.

In [ ]:
class MultiHead(nn.Module):
    def __init__(self, d_model, h):
        super().__init__()
        assert d_model % h == 0
        self.h, self.d_k = h, d_model // h
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        B = q.size(0)
        # YOUR CODE HERE
        raise NotImplementedError

In [ ]:
# TEST — Task 2
mha = MultiHead(d_model=32, h=4)
x = torch.randn(2, 5, 32)
y = mha(x, x, x)
assert y.shape == (2, 5, 32), f"output shape {y.shape}"
y2 = mha(x, x, x, torch.tril(torch.ones(1, 5, 5)).long())
assert not torch.isnan(y2).any(), "masked forward should not produce NaNs"
print("PASS — MultiHead")

## Task 3 — Positional encoding (build the table)

Fill the `pe` table of shape `(max_len, d_model)`:
- `pos` is a column vector of positions `0 … max_len-1`.
- `div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))`.
- **Even** dims get `sin(pos * div)`; **odd** dims get `cos(pos * div)`.
  Hint: `pe[:, 0::2] = ...` and `pe[:, 1::2] = ...`.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        # YOUR CODE HERE
        # div = ...
        # pe[:, 0::2] = ...
        # pe[:, 1::2] = ...
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [ ]:
# TEST — Task 3
pe = PositionalEncoding(16, max_len=50)
assert pe.pe.shape == (1, 50, 16), f"pe shape {pe.pe.shape}"
assert torch.allclose(pe.pe[0, 0, 0::2], torch.zeros(8), atol=1e-6), "sin(0) = 0 on even dims"
assert torch.allclose(pe.pe[0, 0, 1::2], torch.ones(8), atol=1e-6), "cos(0) = 1 on odd dims"
assert pe(torch.randn(2, 7, 16)).shape == (2, 7, 16), "forward must preserve shape"
print("PASS — PositionalEncoding")

## Task 4 — Causal mask

Return a `(1, T, T)` tensor: 1 on and below the diagonal, 0 above (a query may attend
to itself and earlier positions, never later ones). Hint: `torch.tril`.

In [ ]:
def causal_mask(T):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# TEST — Task 4
cm = causal_mask(4)
assert cm.shape == (1, 4, 4), f"shape {cm.shape}"
assert (cm[0].tril() == cm[0]).all(), "must be lower-triangular (no future)"
assert cm[0].sum().item() == 10, "should have 4+3+2+1 = 10 ones"
print("PASS — causal_mask")

## Provided — feed-forward and encoder layer

These are given so you can focus on the decoder next. Just run the cell.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(),
                                 nn.Linear(d_ff, d_model))
    def forward(self, x):
        return self.net(x)


class EncoderLayer(nn.Module):
    def __init__(self, d_model, h, d_ff):
        super().__init__()
        self.attn = MultiHead(d_model, h)
        self.ff = FeedForward(d_model, d_ff)
        self.n1, self.n2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
    def forward(self, x, mask=None):
        x = self.n1(x + self.attn(x, x, x, mask))
        x = self.n2(x + self.ff(x))
        return x

## Task 5 — Decoder layer (`forward`)

The `__init__` is given: `self_attn`, `cross_attn`, `ff`, and three norms. Wire up the
three sublayers, each as `norm(x + sublayer(x))`:
1. **Masked self-attention** — `self_attn(x, x, x, self_mask)`.
2. **Cross-attention** — queries from the decoder, keys/values from the encoder:
   `cross_attn(x, enc, enc, cross_mask)`. *(This is the one place q and k/v differ.)*
3. **Feed-forward** — `ff(x)`.

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, h, d_ff):
        super().__init__()
        self.self_attn = MultiHead(d_model, h)
        self.cross_attn = MultiHead(d_model, h)
        self.ff = FeedForward(d_model, d_ff)
        self.n1, self.n2, self.n3 = (nn.LayerNorm(d_model) for _ in range(3))

    def forward(self, x, enc, self_mask=None, cross_mask=None):
        # YOUR CODE HERE  (three lines: self-attn, cross-attn, feed-forward)
        raise NotImplementedError

In [ ]:
# TEST — Task 5
dl = DecoderLayer(32, h=4, d_ff=64)
tgt, enc = torch.randn(2, 6, 32), torch.randn(2, 5, 32)
o = dl(tgt, enc, causal_mask(6), None)
assert o.shape == (2, 6, 32), f"output shape {o.shape}"
o.sum().backward()
assert dl.self_attn.W_q.weight.grad is not None, "gradients should flow"
print("PASS — DecoderLayer")

## Put it together and train

Nothing to fill in — this uses your five pieces. If they're right, token-accuracy on
the reverse task climbs to ~1.0.

In [ ]:
class Transformer(nn.Module):
    def __init__(self, vocab, d_model=64, h=4, d_ff=128, layers=2, max_len=64):
        super().__init__()
        self.d_model = d_model
        self.emb = nn.Embedding(vocab, d_model)
        self.pos = PositionalEncoding(d_model, max_len)
        self.enc = nn.ModuleList([EncoderLayer(d_model, h, d_ff) for _ in range(layers)])
        self.dec = nn.ModuleList([DecoderLayer(d_model, h, d_ff) for _ in range(layers)])
        self.out = nn.Linear(d_model, vocab)
    def encode(self, src):
        x = self.pos(self.emb(src) * math.sqrt(self.d_model))
        for l in self.enc: x = l(x)
        return x
    def decode(self, tgt, enc):
        x = self.pos(self.emb(tgt) * math.sqrt(self.d_model))
        m = causal_mask(tgt.size(1)).to(tgt.device)
        for l in self.dec: x = l(x, enc, m, None)
        return x
    def forward(self, src, tgt):
        return self.out(self.decode(tgt, self.encode(src)))

DIGITS, BOS, VOCAB, L = 10, 10, 11, 8
def make_batch(n):
    src = torch.randint(0, DIGITS, (n, L)); tgt = torch.flip(src, dims=[1])
    return src, torch.cat([torch.full((n, 1), BOS), tgt[:, :-1]], 1), tgt

@torch.no_grad()
def greedy(model, src):
    enc = model.encode(src); ys = torch.full((src.size(0), 1), BOS)
    for _ in range(L):
        ys = torch.cat([ys, model.out(model.decode(ys, enc)[:, -1]).argmax(-1, keepdim=True)], 1)
    return ys[:, 1:]

model = Transformer(VOCAB); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for step in range(1, 2001):
    s, di, t = make_batch(64)
    loss = F.cross_entropy(model(s, di).reshape(-1, VOCAB), t.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 250 == 0:
        vs, _, vt = make_batch(200)
        print(f"step {step:4d}   loss {loss.item():.3f}   token-acc {(greedy(model, vs) == vt).float().mean():.3f}")

s, _, t = make_batch(1)
print("\ninput   :", s[0].tolist())
print("model   :", greedy(model, s)[0].tolist())
print("expected:", t[0].tolist())

## Stretch (optional)

- **Attention heatmap.** In `MultiHead.forward`, return the weights too and plot them
  for one head — do cross-attention heads line up decoder position *i* with encoder
  position *L-i* (the reversal)?
- **Change the task.** Make the target `sorted(src)` instead of reversed. Does it still
  train? Which is harder?
- **Pre-norm.** Switch the layers to `x + sublayer(norm(x))`. Any difference in training
  speed? (This is the standard modern choice.)
- **Ablate positions.** Zero out the positional encoding. Watch accuracy collapse —
  proof that attention alone is order-blind.